In [11]:
# ============================================================
# ONLINE RETAIL SALES ANALYSIS
# NO PANDAS
# NO DATA DELETION
# ORIGINAL DATA IS PRESERVED
# ============================================================

import csv
import os
import html
from pathlib import Path
from datetime import datetime
from collections import defaultdict


# ============================================================
# 1. CONFIGURATION
# ============================================================

FILE_PATH = "online_retail.csv"

OUTPUT_DIR = Path(
    "retail_analysis_output"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)

print("=" * 80)
print("              ONLINE RETAIL SALES ANALYSIS")
print("              ORIGINAL DATA PRESERVED")
print("=" * 80)


# ============================================================
# 2. CHECK FILE
# ============================================================

if not os.path.exists(FILE_PATH):

    print("\nERROR: online_retail.csv not found.")

    print("\nAvailable files:")

    for file in os.listdir("."):
        print(" -", file)

    raise FileNotFoundError(
        "Please upload online_retail.csv."
    )


# ============================================================
# 3. LOAD ORIGINAL DATA
# ============================================================

with open(
    FILE_PATH,
    "r",
    encoding="utf-8-sig",
    newline=""
) as file:

    reader = csv.DictReader(file)

    original_data = list(reader)

    columns = reader.fieldnames


print(
    "\nOriginal rows:",
    f"{len(original_data):,}"
)

print("\nColumns:")

for column in columns:
    print(" -", column)


# ============================================================
# 4. KEEP ORIGINAL DATA
# ============================================================

# IMPORTANT:
# Nothing is deleted from original_data.

data = []

invalid_quantity = 0
invalid_price = 0
invalid_date = 0
missing_customer = 0
return_transactions = 0


# ============================================================
# 5. PROCESS DATA WITHOUT DELETING ROWS
# ============================================================

for row in original_data:

    # --------------------------------------------------------
    # Quantity
    # --------------------------------------------------------

    try:

        quantity = float(
            row["Quantity"]
        )

    except:

        quantity = 0

        invalid_quantity += 1


    # --------------------------------------------------------
    # Unit Price
    # --------------------------------------------------------

    try:

        unit_price = float(
            row["UnitPrice"]
        )

    except:

        unit_price = 0

        invalid_price += 1


    # --------------------------------------------------------
    # Invoice Date
    # --------------------------------------------------------

    try:

        invoice_date = datetime.strptime(
            row["InvoiceDate"].strip(),
            "%d-%m-%Y %H:%M"
        )

    except:

        invoice_date = None

        invalid_date += 1


    # --------------------------------------------------------
    # Customer
    # --------------------------------------------------------

    customer_id = (
        row["CustomerID"].strip()
        if row["CustomerID"]
        else ""
    )

    if not customer_id:

        missing_customer += 1


    # --------------------------------------------------------
    # Product
    # --------------------------------------------------------

    description = (
        row["Description"].strip()
        if row["Description"]
        else "Unknown Product"
    )


    # --------------------------------------------------------
    # Country
    # --------------------------------------------------------

    country = (
        row["Country"].strip()
        if row["Country"]
        else "Unknown"
    )


    # --------------------------------------------------------
    # Revenue
    # --------------------------------------------------------

    revenue = (
        quantity *
        unit_price
    )


    # --------------------------------------------------------
    # Transaction Type
    # --------------------------------------------------------

    if quantity < 0:

        transaction_type = "Return"

        return_transactions += 1

    elif quantity == 0:

        transaction_type = "Zero Quantity"

    elif unit_price <= 0:

        transaction_type = "Invalid Price"

    else:

        transaction_type = "Sale"


    # --------------------------------------------------------
    # Date information
    # --------------------------------------------------------

    if invoice_date:

        month = invoice_date.strftime(
            "%Y-%m"
        )

        year = invoice_date.year

        quarter = (
            "Q" +
            str(
                ((invoice_date.month - 1) // 3) + 1
            )
        )

        date = invoice_date.strftime(
            "%Y-%m-%d"
        )

    else:

        month = "Unknown"
        year = "Unknown"
        quarter = "Unknown"
        date = "Unknown"


    # --------------------------------------------------------
    # Store processed row
    # --------------------------------------------------------

    processed_row = dict(row)

    processed_row["CalculatedRevenue"] = revenue

    processed_row["TransactionType"] = (
        transaction_type
    )

    processed_row["Month"] = month

    processed_row["Year"] = year

    processed_row["Quarter"] = quarter

    processed_row["Date"] = date

    data.append(
        processed_row
    )


# ============================================================
# 6. DATA QUALITY REPORT
# ============================================================

print("\n" + "=" * 80)
print("DATA QUALITY")
print("=" * 80)

print(
    "\nOriginal records:",
    f"{len(original_data):,}"
)

print(
    "Records retained:",
    f"{len(data):,}"
)

print(
    "Records deleted:",
    "0"
)

print(
    "Return transactions:",
    f"{return_transactions:,}"
)

print(
    "Invalid quantity:",
    f"{invalid_quantity:,}"
)

print(
    "Invalid price:",
    f"{invalid_price:,}"
)

print(
    "Invalid dates:",
    f"{invalid_date:,}"
)

print(
    "Missing CustomerID:",
    f"{missing_customer:,}"
)


# ============================================================
# 7. CREATE SALES-ONLY VIEW
# ============================================================

# The original data remains untouched.
#
# For revenue/business analysis we use only valid positive
# sales transactions.

sales_data = []

for row in data:

    try:

        quantity = float(
            row["Quantity"]
        )

        unit_price = float(
            row["UnitPrice"]
        )

    except:

        continue


    if (
        quantity > 0
        and
        unit_price > 0
        and
        row["TransactionType"] == "Sale"
    ):

        sales_data.append(row)


print(
    "\nRecords used for sales analysis:",
    f"{len(sales_data):,}"
)

print(
    "Original dataset records:",
    f"{len(data):,}"
)


# ============================================================
# 8. KPI ANALYSIS
# ============================================================

print("\n" + "=" * 80)
print("KEY PERFORMANCE INDICATORS")
print("=" * 80)


total_revenue = 0
total_units = 0

orders = set()
customers = set()
countries = set()


for row in sales_data:

    quantity = float(
        row["Quantity"]
    )

    revenue = float(
        row["CalculatedRevenue"]
    )

    total_revenue += revenue

    total_units += quantity

    orders.add(
        row["InvoiceNo"]
    )

    if row["CustomerID"]:

        customers.add(
            row["CustomerID"]
        )

    countries.add(
        row["Country"]
    )


total_orders = len(
    orders
)

unique_customers = len(
    customers
)

number_of_countries = len(
    countries
)


average_order_value = (
    total_revenue /
    total_orders
    if total_orders
    else 0
)


print(
    f"\nTotal Revenue       : "
    f"£{total_revenue:,.2f}"
)

print(
    f"Total Orders        : "
    f"{total_orders:,}"
)

print(
    f"Unique Customers    : "
    f"{unique_customers:,}"
)

print(
    f"Countries           : "
    f"{number_of_countries:,}"
)

print(
    f"Total Units Sold    : "
    f"{total_units:,.0f}"
)

print(
    f"Average Order Value : "
    f"£{average_order_value:,.2f}"
)


# ============================================================
# 9. MONTHLY SALES
# ============================================================

monthly = defaultdict(
    lambda: {
        "revenue": 0,
        "units": 0,
        "orders": set(),
        "customers": set()
    }
)


for row in sales_data:

    month = row["Month"]

    monthly[month]["revenue"] += (
        float(
            row["CalculatedRevenue"]
        )
    )

    monthly[month]["units"] += (
        float(
            row["Quantity"]
        )
    )

    monthly[month]["orders"].add(
        row["InvoiceNo"]
    )

    if row["CustomerID"]:

        monthly[month]["customers"].add(
            row["CustomerID"]
        )


monthly_results = []

previous_revenue = None


for month in sorted(
    monthly.keys()
):

    revenue = (
        monthly[month]["revenue"]
    )

    orders_count = len(
        monthly[month]["orders"]
    )

    units = (
        monthly[month]["units"]
    )

    customers_count = len(
        monthly[month]["customers"]
    )


    if previous_revenue is None:

        growth = ""

    elif previous_revenue == 0:

        growth = ""

    else:

        growth = (
            (
                revenue -
                previous_revenue
            )
            /
            previous_revenue
            *
            100
        )


    revenue_share = (
        revenue /
        total_revenue *
        100
        if total_revenue
        else 0
    )


    monthly_results.append([
        month,
        revenue,
        orders_count,
        units,
        customers_count,
        growth,
        revenue_share
    ])


    previous_revenue = revenue


print("\nMONTHLY REVENUE")

print("-" * 70)

for item in monthly_results:

    print(
        f"{item[0]:<12}"
        f"£{item[1]:>15,.2f}"
        f"{item[2]:>12,}"
        f"{item[3]:>15,.0f}"
    )


# ============================================================
# 10. PEAK MONTH
# ============================================================

peak_month = max(
    monthly_results,
    key=lambda x: x[1]
)


print(
    "\nPeak Month:",
    peak_month[0]
)

print(
    "Peak Revenue:",
    f"£{peak_month[1]:,.2f}"
)


# ============================================================
# 11. PRODUCT ANALYSIS
# ============================================================

products = defaultdict(
    lambda: {
        "revenue": 0,
        "units": 0,
        "orders": set(),
        "customers": set()
    }
)


for row in sales_data:

    key = (
        row["StockCode"],
        row["Description"]
    )

    products[key]["revenue"] += (
        float(
            row["CalculatedRevenue"]
        )
    )

    products[key]["units"] += (
        float(
            row["Quantity"]
        )
    )

    products[key]["orders"].add(
        row["InvoiceNo"]
    )

    if row["CustomerID"]:

        products[key]["customers"].add(
            row["CustomerID"]
        )


product_results = []


for key, value in products.items():

    revenue = value["revenue"]

    units = value["units"]

    order_count = len(
        value["orders"]
    )

    customer_count = len(
        value["customers"]
    )

    share = (
        revenue /
        total_revenue *
        100
        if total_revenue
        else 0
    )


    product_results.append([

        key[0],

        key[1],

        revenue,

        units,

        order_count,

        customer_count,

        share
    ])


product_results.sort(
    key=lambda x: x[2],
    reverse=True
)


print("\n" + "=" * 80)
print("TOP 10 PRODUCTS")
print("=" * 80)


for i, product in enumerate(
    product_results[:10],
    1
):

    print(
        f"{i:>2}. "
        f"{str(product[1])[:45]:<45}"
        f"£{product[2]:>14,.2f}"
    )


top_product = (
    product_results[0]
)


# ============================================================
# 12. COUNTRY ANALYSIS
# ============================================================

countries_data = defaultdict(
    lambda: {
        "revenue": 0,
        "units": 0,
        "orders": set(),
        "customers": set()
    }
)


for row in sales_data:

    country = row["Country"]

    countries_data[country]["revenue"] += (
        float(
            row["CalculatedRevenue"]
        )
    )

    countries_data[country]["units"] += (
        float(
            row["Quantity"]
        )
    )

    countries_data[country]["orders"].add(
        row["InvoiceNo"]
    )

    if row["CustomerID"]:

        countries_data[country]["customers"].add(
            row["CustomerID"]
        )


country_results = []


for country, value in countries_data.items():

    revenue = value["revenue"]

    units = value["units"]

    order_count = len(
        value["orders"]
    )

    customer_count = len(
        value["customers"]
    )

    share = (
        revenue /
        total_revenue *
        100
        if total_revenue
        else 0
    )


    country_results.append([

        country,

        revenue,

        order_count,

        units,

        customer_count,

        share
    ])


country_results.sort(
    key=lambda x: x[1],
    reverse=True
)


print("\n" + "=" * 80)
print("TOP 10 COUNTRIES")
print("=" * 80)


for i, country in enumerate(
    country_results[:10],
    1
):

    print(
        f"{i:>2}. "
        f"{country[0]:<25}"
        f"£{country[1]:>14,.2f}"
    )


top_country = (
    country_results[0]
)


# ============================================================
# 13. CUSTOMER ANALYSIS
# ============================================================

customer_data = defaultdict(
    lambda: {
        "revenue": 0,
        "units": 0,
        "orders": set()
    }
)


for row in sales_data:

    customer = row["CustomerID"]

    if not customer:
        continue


    customer_data[customer]["revenue"] += (
        float(
            row["CalculatedRevenue"]
        )
    )

    customer_data[customer]["units"] += (
        float(
            row["Quantity"]
        )
    )

    customer_data[customer]["orders"].add(
        row["InvoiceNo"]
    )


customer_results = []


for customer, value in customer_data.items():

    revenue = value["revenue"]

    units = value["units"]

    order_count = len(
        value["orders"]
    )


    customer_results.append([

        customer,

        revenue,

        order_count,

        units
    ])


customer_results.sort(
    key=lambda x: x[1],
    reverse=True
)


print("\n" + "=" * 80)
print("TOP 10 CUSTOMERS")
print("=" * 80)


for i, customer in enumerate(
    customer_results[:10],
    1
):

    print(
        f"{i:>2}. "
        f"{customer[0]:<15}"
        f"£{customer[1]:>14,.2f}"
    )


# ============================================================
# 14. Q4 ANALYSIS
# ============================================================

q4_revenue = 0


for row in sales_data:

    if row["Quarter"] == "Q4":

        q4_revenue += (
            float(
                row["CalculatedRevenue"]
            )
        )


q4_share = (
    q4_revenue /
    total_revenue *
    100
    if total_revenue
    else 0
)


print("\n" + "=" * 80)
print("Q4 ANALYSIS")
print("=" * 80)

print(
    f"Q4 Revenue: "
    f"£{q4_revenue:,.2f}"
)

print(
    f"Q4 Revenue Share: "
    f"{q4_share:.2f}%"
)


# ============================================================
# 15. BUSINESS INSIGHTS
# ============================================================

insights = [

    (
        "Peak Sales Month",
        f"{peak_month[0]} generated "
        f"£{peak_month[1]:,.2f}."
    ),

    (
        "Top Product",
        f"{top_product[1]} generated "
        f"£{top_product[2]:,.2f}."
    ),

    (
        "Top Country",
        f"{top_country[0]} generated "
        f"£{top_country[1]:,.2f}."
    ),

    (
        "Q4 Contribution",
        f"Q4 contributed "
        f"{q4_share:.2f}% "
        f"of total revenue."
    ),

    (
        "Average Order Value",
        f"The average order value was "
        f"£{average_order_value:,.2f}."
    ),

    (
        "Data Preservation",
        f"All {len(original_data):,} "
        f"original records were retained."
    )
]


print("\n" + "=" * 80)
print("BUSINESS INSIGHTS")
print("=" * 80)


for i, (
    title,
    description
) in enumerate(
    insights,
    1
):

    print(
        f"\n{i}. {title}"
    )

    print(
        "   " +
        description
    )


# ============================================================
# 16. RECOMMENDATIONS
# ============================================================

recommendations = [

    (
        "Seasonal Planning",
        "Prepare inventory and marketing campaigns "
        "before high-revenue periods."
    ),

    (
        "Product Management",
        "Monitor high-revenue products for stock "
        "availability and demand changes."
    ),

    (
        "Customer Retention",
        "Identify repeat and high-value customers "
        "for targeted campaigns."
    ),

    (
        "Geographic Analysis",
        "Compare markets by revenue, orders and "
        "customer activity."
    ),

    (
        "Return Monitoring",
        "Track return transactions separately to "
        "understand their effect on sales."
    ),

    (
        "RFM Analysis",
        "Perform Recency, Frequency and Monetary "
        "segmentation."
    )
]


# ============================================================
# 17. EXPORT ORIGINAL DATA
# ============================================================

print("\n" + "=" * 80)
print("EXPORTING FILES")
print("=" * 80)


# Original data is copied without deleting anything.

original_output = (
    OUTPUT_DIR /
    "original_data_preserved.csv"
)


with open(
    original_output,
    "w",
    encoding="utf-8",
    newline=""
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=columns
    )

    writer.writeheader()

    writer.writerows(
        original_data
    )


print(
    "Original dataset preserved:"
)

print(
    original_output
)


# ============================================================
# 18. EXPORT MONTHLY DATA
# ============================================================

def save_csv(
    filename,
    headers,
    rows_to_save
):

    path = (
        OUTPUT_DIR /
        filename
    )

    with open(
        path,
        "w",
        encoding="utf-8",
        newline=""
    ) as file:

        writer = csv.writer(file)

        writer.writerow(
            headers
        )

        writer.writerows(
            rows_to_save
        )

    return path


save_csv(

    "monthly_sales.csv",

    [
        "Month",
        "Revenue",
        "Orders",
        "Units",
        "Customers",
        "Growth_%",
        "Revenue_Share_%"
    ],

    monthly_results
)


# ============================================================
# 19. EXPORT PRODUCT DATA
# ============================================================

save_csv(

    "product_analysis.csv",

    [
        "StockCode",
        "Description",
        "Revenue",
        "Units",
        "Orders",
        "Customers",
        "Revenue_Share_%"
    ],

    product_results
)


# ============================================================
# 20. EXPORT COUNTRY DATA
# ============================================================

save_csv(

    "country_analysis.csv",

    [
        "Country",
        "Revenue",
        "Orders",
        "Units",
        "Customers",
        "Revenue_Share_%"
    ],

    country_results
)


# ============================================================
# 21. EXPORT CUSTOMER DATA
# ============================================================

save_csv(

    "customer_analysis.csv",

    [
        "CustomerID",
        "Revenue",
        "Orders",
        "Units"
    ],

    customer_results
)


# ============================================================
# 22. EXPORT DATA QUALITY
# ============================================================

save_csv(

    "data_quality_report.csv",

    [
        "Metric",
        "Value"
    ],

    [

        [
            "Original Records",
            len(original_data)
        ],

        [
            "Records Retained",
            len(data)
        ],

        [
            "Records Deleted",
            0
        ],

        [
            "Records Used for Sales Analysis",
            len(sales_data)
        ],

        [
            "Return Transactions",
            return_transactions
        ],

        [
            "Invalid Quantity",
            invalid_quantity
        ],

        [
            "Invalid Price",
            invalid_price
        ],

        [
            "Invalid Dates",
            invalid_date
        ],

        [
            "Missing CustomerID",
            missing_customer
        ]
    ]
)


# ============================================================
# 23. EXPORT KPI DATA
# ============================================================

save_csv(

    "kpi_summary.csv",

    [
        "Metric",
        "Value"
    ],

    [

        [
            "Total Revenue",
            total_revenue
        ],

        [
            "Total Orders",
            total_orders
        ],

        [
            "Unique Customers",
            unique_customers
        ],

        [
            "Countries",
            number_of_countries
        ],

        [
            "Total Units",
            total_units
        ],

        [
            "Average Order Value",
            average_order_value
        ],

        [
            "Q4 Revenue",
            q4_revenue
        ],

        [
            "Q4 Revenue Share %",
            q4_share
        ]
    ]
)


# ============================================================
# 24. EXPORT INSIGHTS
# ============================================================

save_csv(

    "business_insights.csv",

    [
        "No",
        "Insight",
        "Description"
    ],

    [
        [
            i,
            title,
            description
        ]

        for i, (
            title,
            description
        )

        in enumerate(
            insights,
            1
        )
    ]
)


# ============================================================
# 25. EXPORT RECOMMENDATIONS
# ============================================================

save_csv(

    "business_recommendations.csv",

    [
        "No",
        "Recommendation",
        "Description"
    ],

    [
        [
            i,
            title,
            description
        ]

        for i, (
            title,
            description
        )

        in enumerate(
            recommendations,
            1
        )
    ]
)


# ============================================================
# 26. FINAL REPORT
# ============================================================

report_path = (
    OUTPUT_DIR /
    "Online_Retail_Report.txt"
)


with open(
    report_path,
    "w",
    encoding="utf-8"
) as report:

    report.write(
        "ONLINE RETAIL SALES ANALYSIS\n"
    )

    report.write(
        "=" * 80 +
        "\n\n"
    )

    report.write(
        "DATA PRESERVATION\n"
    )

    report.write(
        "-" * 80 +
        "\n"
    )

    report.write(
        f"Original records: "
        f"{len(original_data):,}\n"
    )

    report.write(
        f"Records deleted: 0\n"
    )

    report.write(
        "All original records were retained.\n\n"
    )


    report.write(
        "KEY PERFORMANCE INDICATORS\n"
    )

    report.write(
        "-" * 80 +
        "\n"
    )

    report.write(
        f"Total Revenue: "
        f"£{total_revenue:,.2f}\n"
    )

    report.write(
        f"Total Orders: "
        f"{total_orders:,}\n"
    )

    report.write(
        f"Unique Customers: "
        f"{unique_customers:,}\n"
    )

    report.write(
        f"Countries: "
        f"{number_of_countries:,}\n"
    )

    report.write(
        f"Average Order Value: "
        f"£{average_order_value:,.2f}\n\n"
    )


    report.write(
        "BUSINESS INSIGHTS\n"
    )

    report.write(
        "-" * 80 +
        "\n"
    )

    for i, (
        title,
        description
    ) in enumerate(
        insights,
        1
    ):

        report.write(
            f"{i}. {title}: "
            f"{description}\n"
        )


    report.write(
        "\nBUSINESS RECOMMENDATIONS\n"
    )

    report.write(
        "-" * 80 +
        "\n"
    )

    for i, (
        title,
        description
    ) in enumerate(
        recommendations,
        1
    ):

        report.write(
            f"{i}. {title}: "
            f"{description}\n"
        )


# ============================================================
# 27. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 80)
print("                 ANALYSIS COMPLETE")
print("=" * 80)

print(
    "\nIMPORTANT:"
)

print(
    "No original data rows were deleted."
)

print(
    f"Original records : "
    f"{len(original_data):,}"
)

print(
    f"Records retained : "
    f"{len(data):,}"
)

print(
    "\nTotal Revenue:",
    f"£{total_revenue:,.2f}"
)

print(
    "Total Orders:",
    f"{total_orders:,}"
)

print(
    "Unique Customers:",
    f"{unique_customers:,}"
)

print(
    "Top Product:",
    top_product[1]
)

print(
    "Top Country:",
    top_country[0]
)

print(
    "\nOutput folder:"
)

print(
    OUTPUT_DIR.resolve()
)

print("\nFiles created:")

for filename in sorted(
    os.listdir(OUTPUT_DIR)
):

    print(
        "✓",
        filename
    )

print("\n" + "=" * 80)
print("                 PROJECT COMPLETED")
print("=" * 80)

              ONLINE RETAIL SALES ANALYSIS
              ORIGINAL DATA PRESERVED

Original rows: 541,909

Columns:
 - InvoiceNo
 - StockCode
 - Description
 - Quantity
 - InvoiceDate
 - UnitPrice
 - CustomerID
 - Country

DATA QUALITY

Original records: 541,909
Records retained: 541,909
Records deleted: 0
Return transactions: 10,624
Invalid quantity: 0
Invalid price: 0
Invalid dates: 541,909
Missing CustomerID: 135,080

Records used for sales analysis: 530,104
Original dataset records: 541,909

KEY PERFORMANCE INDICATORS

Total Revenue       : £10,666,684.54
Total Orders        : 19,960
Unique Customers    : 4,338
Countries           : 38
Total Units Sold    : 5,588,376
Average Order Value : £534.40

MONTHLY REVENUE
----------------------------------------------------------------------
Unknown     £  10,666,684.54      19,960      5,588,376

Peak Month: Unknown
Peak Revenue: £10,666,684.54

TOP 10 PRODUCTS
 1. DOTCOM POSTAGE                               £    206,248.77
 2. REGENCY CAK

In [12]:
import zipfile
from pathlib import Path

folder = Path("retail_analysis_output")
zip_file = Path("retail_analysis_output.zip")

if not folder.exists():
    print("ERROR: retail_analysis_output folder not found.")
else:
    # Delete old ZIP if it exists
    if zip_file.exists():
        zip_file.unlink()

    # Add EVERYTHING inside the retail folder
    with zipfile.ZipFile(zip_file, "w", zipfile.ZIP_DEFLATED) as z:
        for file in folder.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(folder.parent))

    files = [f for f in folder.rglob("*") if f.is_file()]

    print("====================================")
    print("ZIP CREATED SUCCESSFULLY")
    print("====================================")
    print("ZIP:", zip_file)
    print("Total files:", len(files))

    print("\nIncluded files:")
    for f in files:
        print(" -", f.relative_to(folder))

ZIP CREATED SUCCESSFULLY
ZIP: retail_analysis_output.zip
Total files: 15

Included files:
 - Online_Retail_Report.txt
 - business_insights.csv
 - business_recommendations.csv
 - cleaned_online_retail.csv
 - country_analysis.csv
 - customer_analysis.csv
 - data_quality_report.csv
 - kpi_summary.csv
 - monthly_revenue_trend.png
 - monthly_sales.csv
 - monthly_sales_analysis.csv
 - original_data_preserved.csv
 - product_analysis.csv
 - top_countries.png
 - top_products.png
